# Water Potability Prediction – End‑to‑End Data Science Notebook


This notebook contains a **complete, production‑grade data science workflow** for predicting water potability.
Steps include:
1. Data loading
2. Exploratory Data Analysis (EDA)
3. Data preprocessing
4. Model training
5. Model evaluation
6. Feature importance analysis


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier


## Load Dataset

In [ ]:

df = pd.read_csv('water_potability.csv')
print(df.shape)
df.head()


## Basic Data Inspection

In [ ]:

df.info()
df.describe()


## Missing Values

In [ ]:

df.isnull().sum()


## Target Distribution

In [ ]:

sns.countplot(x='Potability', data=df)
plt.title('Target Distribution')
plt.show()


## Feature Distributions

In [ ]:

df.hist(bins=30, figsize=(16,12))
plt.suptitle('Feature Distributions')
plt.show()


## Correlation Heatmap

In [ ]:

plt.figure(figsize=(12,8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()


## Train/Test Split

In [ ]:

X = df.drop('Potability', axis=1)
y = df['Potability']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)


## Baseline Model: Logistic Regression

In [ ]:

log_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

log_pipeline.fit(X_train, y_train)
preds = log_pipeline.predict(X_test)
proba = log_pipeline.predict_proba(X_test)[:,1]

print('Accuracy:', accuracy_score(y_test, preds))
print('ROC AUC:', roc_auc_score(y_test, proba))
print(classification_report(y_test, preds))


## Advanced Model: Random Forest

In [ ]:

rf_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('model', RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight='balanced'
    ))
])

rf_pipeline.fit(X_train, y_train)
rf_preds = rf_pipeline.predict(X_test)
rf_proba = rf_pipeline.predict_proba(X_test)[:,1]

print('Accuracy:', accuracy_score(y_test, rf_preds))
print('ROC AUC:', roc_auc_score(y_test, rf_proba))
print(classification_report(y_test, rf_preds))


## Confusion Matrix

In [ ]:

cm = confusion_matrix(y_test, rf_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix – Random Forest')
plt.show()


## Feature Importance

In [ ]:

rf_model = rf_pipeline.named_steps['model']
importances = pd.Series(rf_model.feature_importances_, index=X.columns)
importances.sort_values(ascending=False).plot(kind='bar', figsize=(12,6))
plt.title('Random Forest Feature Importance')
plt.show()


## Cross‑Validation ROC‑AUC

In [ ]:

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(rf_pipeline, X, y, cv=skf, scoring='roc_auc')

print('Cross‑validated ROC‑AUC:', cv_scores)
print('Mean ROC‑AUC:', cv_scores.mean())


## Final Conclusion


✅ Random Forest provides the best balance of performance and interpretability.

Key drivers of water potability:
- Solids
- Sulfate
- Organic Carbon
- pH

This model is suitable for deployment after threshold tuning and monitoring.
